<a href="https://colab.research.google.com/github/KSamar33/samar-codeboosters-2026/blob/main/Day%203/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')
print(f'pandas : {pd.__version__}')
print(f'requests : {requests.__version__}')

All libraries imported successfully!
pandas : 2.2.2
requests : 2.32.4


In [26]:
raw_df=pd.read_csv('messy_sales_data.csv')

print(f'Raw data loaded:{raw_df.shape[0]}rows,{raw_df.shape[1]}columns')
print(f'columns:{raw_df.columns.tolist()}')
print('\nFirst 5 rows:')
raw_df.head()

Raw data loaded:30rows,9columns
columns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 rows:


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [27]:
print('=' * 55)
print('DATA QUALITY DIAGANOSIS REPORT')
print('=' * 55)

print('\n[1] MISSING VALUES per column:')
print(raw_df.isnull().sum())

print(f'\n[2] DUPLICATE ROWS: {raw_df.duplicated().sum()}')

print('\n[3] DATA TYPES:')
print(raw_df.dtypes)

print('\n[4] UNIQUE CATEGORIES:', raw_df['category'].unique())
print('[4] Sample customer names:', raw_df['customer_name'].dropna().unique()[:8])
print('[4] Smaple order_date values:', raw_df['order_date'].unique()[:6])

DATA QUALITY DIAGANOSIS REPORT

[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS: 0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Smaple order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [30]:
print('=' * 55)
print('DATA QUALITY DIAGANOSIS REPORT')
print('=' * 55)

print('\n[1] MISSING VALUES in quantity:')
print(raw_df['quantity'].isnull().sum())

DATA QUALITY DIAGANOSIS REPORT

[1] MISSING VALUES in quantity:
3


In [31]:
df = raw_df.copy()
print(f'Working copy created: {df.shape}')
print('raw_df is untouched - we can always reset by running df = raw_df.copy()')

Working copy created: (30, 9)
raw_df is untouched - we can always reset by running df = raw_df.copy()


In [32]:
print('Before fixing nulls:',df.isnull().sum().sum(),'total missing values')
df['customer_name'].fillna('Unknown Customers',inplace=True)
median_qty= df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f'Filled missing) quantity with median:{median_qty}')
df['category'].fillna('uncategorized',inplace=True)
df['product'].fillna('Unknown Product',inplace=True)

print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 7 total missing values
Filled missing) quantity with median:2.0
After fixing nulls: 0 total missing values


In [33]:
print(f'Before deduplication: {len(df)}rows')
print(f'Duplicate rows:{df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
df.drop_duplicates(inplace=True)
print(f'\nAfter deduplication: {len(df)}rows')
print(f'Rows removed:{len(raw_df)}-len(df)')

Before deduplication: 30rows
Duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication: 30rows
Rows removed:30-len(df)


In [34]:
df['product'].fillna('NotListed', inplace = True)
print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')
#====================================
#Fix #2: Remove Duplicates
#====================================
print(f'Before duplication :{len(df)} rows')
print(f'Duplicate rows:{df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name', 'product', 'order_id']])
df.drop_duplicates(inplace=True)
print(f'\nAfter deduplication: {len(df)} rows')
print(f'Rows removed: {len(raw_df) - len(df)}')

After fixing nulls: 0 total missing values
Before duplication :30 rows
Duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_id]
Index: []

After deduplication: 30 rows
Rows removed: 0


In [35]:
print('Sample data before parsing:')
print(df['order_date'].head(8).tolist())

df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)
nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%B')

print('\nSample data after parsing:')
print(df[['order_date','year','month','month_name']].head(5))

Sample data before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparseable dates (NaT): 2

Sample data after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
